<center>
    <a href="https://www.aus.edu/"><img src="https://i.imgur.com/pdZvnSD.png" width=200> </a>    
</center>
<h1 align=center><font size = 5>CMP 49412 - Personalized Recommendations</font>
<h1 align=center><font size = 5>Item-based Collaborative Filtering</font>
<h1 align=center><font size = 5><b>Name:</b> Mohamed Alawadhi</font>
<h1 align=center><font size = 5><b>ID:</b> b00094286</font>

In [1]:
# Importing the necessary libraries for this assignment
import numpy as np
import pandas as pd
import math

In [2]:
# Read the csv file
data_df = pd.read_csv('rating.csv')

In [3]:
# Print out the first 5 entires of the dataset
data_df.head()

,user_id,movie_id,rating
0,User7307,Movie10,4.5
1,User7307,Movie68,2.5
2,User7307,Movie143,3.5
3,User7307,Movie19,5.0
4,User7307,Movie85,4.5


In [4]:
# Printing out the shape and the number of unique users and movies in the dataset
print(f'The shape of the dataset: {data_df.shape} \n')
print(f'There is a total of "{data_df['user_id'].nunique()}" users and "{data_df['movie_id'].nunique()}" movies in our dataset.')

The shape of the dataset: (5392025, 3) 

There is a total of "10000" users and "2000" movies in our dataset.


**Part 1**
---

In [5]:
user_to_movie = data_df.groupby('user_id')['movie_id'].apply(list).to_dict()

In [6]:
# Displays the list of movies that "User0" has rated
user_to_movie['User0']

['Movie10',
 'Movie125',
 'Movie328',
 'Movie1646',
 'Movie359',
 'Movie104',
 'Movie319',
 'Movie1242',
 'Movie68',
 'Movie186',
 'Movie828',
 'Movie197',
 'Movie148',
 'Movie964',
 'Movie143',
 'Movie1206',
 'Movie103',
 'Movie465',
 'Movie1170',
 'Movie621',
 'Movie136',
 'Movie1647',
 'Movie1498',
 'Movie564',
 'Movie492',
 'Movie19',
 'Movie50',
 'Movie144',
 'Movie85',
 'Movie1072',
 'Movie1779',
 'Movie1740',
 'Movie422',
 'Movie530',
 'Movie1479',
 'Movie23',
 'Movie317',
 'Movie14',
 'Movie496',
 'Movie1791',
 'Movie385',
 'Movie790',
 'Movie1875',
 'Movie170',
 'Movie1979',
 'Movie1161',
 'Movie330',
 'Movie1557',
 'Movie1370',
 'Movie1419',
 'Movie898',
 'Movie1477',
 'Movie1746',
 'Movie1668',
 'Movie1154',
 'Movie1036',
 'Movie1465',
 'Movie1866',
 'Movie1222',
 'Movie127',
 'Movie1201',
 'Movie1159',
 'Movie189',
 'Movie481',
 'Movie783',
 'Movie6',
 'Movie99',
 'Movie370',
 'Movie1257',
 'Movie1861',
 'Movie1369',
 'Movie1958',
 'Movie793',
 'Movie988',
 'Movie129',
 'Mo

In [7]:
movie_to_user = data_df.groupby('movie_id')['user_id'].apply(list).to_dict()

In [8]:
# The list of users who have rated "Movie0"
movie_to_user['Movie0']

['User7307',
 'User7257',
 'User3863',
 'User4473',
 'User4358',
 'User8832',
 'User1900',
 'User1404',
 'User9442',
 'User5557',
 'User190',
 'User926',
 'User5192',
 'User4998',
 'User9674',
 'User4502',
 'User1342',
 'User3223',
 'User901',
 'User3566',
 'User4189',
 'User3584',
 'User1106',
 'User3892',
 'User8352',
 'User5593',
 'User9926',
 'User9611',
 'User8224',
 'User6295',
 'User8564',
 'User849',
 'User661',
 'User2511',
 'User5284',
 'User1698',
 'User4329',
 'User1466',
 'User3412',
 'User2320',
 'User9551',
 'User9519',
 'User2997',
 'User2063',
 'User4411',
 'User7371',
 'User7659',
 'User176',
 'User6318',
 'User348',
 'User268',
 'User2839',
 'User5360',
 'User1582',
 'User7341',
 'User6437',
 'User7103',
 'User5533',
 'User97',
 'User1678',
 'User8098',
 'User3335',
 'User6987',
 'User7788',
 'User1553',
 'User188',
 'User5501',
 'User3783',
 'User7593',
 'User3354',
 'User7407',
 'User7512',
 'User9234',
 'User6137',
 'User7408',
 'User2907',
 'User8437',
 'User2187

In [9]:
user_movie = zip(data_df['user_id'], data_df['movie_id'])
user_movie_rating = zip(user_movie, data_df['rating'])
user_movie_to_rating = dict(user_movie_rating)

In [10]:
user_movie_to_rating[('User0', 'Movie0')]

4.5

In [11]:
target_user = 'User25'

**Part 2**
---

<h1 align=left><font size = 5><b>Adjusted Cosine Similarity</b></font>

$$
\text{sim}(i, j) = \frac{\sum_{u \in U_{i, j}} \left( r_{u, i} - \mu_{u} \right) \left( r_{u, j} - \mu_{u} \right)}
{\sqrt{\sum_{u \in U_{i, j}} \left( r_{u, i} - \mu_{u} \right)^2} \sqrt{\sum_{u \in U_{i, j}} \left( r_{u, j} - \mu_{u} \right)^2}}
$$

where:  


- $sim(𝑖,𝑗)$: The similarity between items 𝑖 and 𝑗.

- $U_{i, j}$: The set of users who have rated both items 𝑖 and 𝑗.

- $𝑟_{𝑢,𝑖}$: The rating given by user 𝑢 to item 𝑖.

- $𝜇_{𝑢}$: The average rating of user 𝑢.

In [12]:
# Calculate the mean of each user
global_means = {}
for user, movies in user_to_movie.items():
    ratings = []
    for movie in movies:
        if (user, movie) in user_movie_to_rating:
            ratings.append(user_movie_to_rating[(user, movie)])
    
    if ratings:
        global_means[user] = sum(ratings) / len(ratings)
    else:
        global_means[user] = 0 

In [13]:
# Get the globel mean of the target_user
mu_target = global_means[target_user]
print(f'Global mean of {target_user}: {mu_target}')

Global mean of User25: 3.66437802907916


In [43]:
# Calculate the mean of each movie
movie_means = {}
for movie, users in movie_to_user.items():
    ratings = []
    for user in users:
        if (user, movie) in user_movie_to_rating:
            ratings.append(user_movie_to_rating[(user, movie)])
        
    if ratings:
        movie_means[movie] = sum(ratings) / len(ratings)
    else:
        movie_means[movie] = 0

In [44]:
print(f'Movie mean of Movie0: {movie_means['Movie0']}')

Movie mean of Movie0: 4.256764342651847


In [20]:
# Movies rated by target_user
target_user_rated_movies = set(user_to_movie[target_user])
print(f'Target user has rated a total of "{len(target_user_rated_movies)}" movies.')

Target user has rated a total of "1238" movies.


In [36]:
# Get the movies that the target user has not rated
movies = set(movie_to_user.keys())
target_user_unrated_movies = movies - target_user_rated_movies
print(f'Number of candidate movies for target user: {len(target_user_unrated_movies)}')

Number of candidate movies for target user: 762


In [ ]:
# Change it into a list
movies = list(movie_to_user.keys())

In [32]:
min_common = 5
similarity_scores = {}

In [ ]:
# Calculate Adjusted Cosine Similarity between movies
# (Note: code execution took ~2 hours to fully execute)
for other_movie in range(len(movies)):

    movie_i = movies[other_movie]
    users_i = set(movie_to_user[movie_i])

    for j in range(other_movie + 1, len(movies)):
        movie_j = movies[j]
        users_j = set(movie_to_user[movie_j])

        common_users = users_i.intersection(users_j)

        if len(common_users) < min_common:
            continue

        numerator = 0
        denominator_i = 0
        denominator_j = 0

        for user in common_users:
            rating_ui = user_movie_to_rating[(user, movie_i)]
            rating_uj = user_movie_to_rating[(user, movie_j)]
            mu_u = global_means[user]
            numerator += (rating_ui - mu_u) * (rating_uj - mu_u)
            denominator_i += (rating_ui - mu_u) ** 2
            denominator_j += (rating_uj - mu_u) ** 2

        if denominator_i == 0 or denominator_j == 0:
            continue

        similarity = numerator / (math.sqrt(denominator_i) * math.sqrt(denominator_j))
        similarity_scores[(movie_i, movie_j)] = similarity
        similarity_scores[(movie_j, movie_i)] = similarity

In [39]:
similarity_scores

{('Movie0', 'Movie1'): 0.30179721272187987,
 ('Movie1', 'Movie0'): 0.30179721272187987,
 ('Movie0', 'Movie10'): 0.3011001026090599,
 ('Movie10', 'Movie0'): 0.3011001026090599,
 ('Movie0', 'Movie100'): 0.3076071198518099,
 ('Movie100', 'Movie0'): 0.3076071198518099,
 ('Movie0', 'Movie1000'): -0.24811789505345472,
 ('Movie1000', 'Movie0'): -0.24811789505345472,
 ('Movie0', 'Movie1001'): -0.3490357047495456,
 ('Movie1001', 'Movie0'): -0.3490357047495456,
 ('Movie0', 'Movie1002'): -0.045177812939195476,
 ('Movie1002', 'Movie0'): -0.045177812939195476,
 ('Movie0', 'Movie1003'): -0.5382970074196066,
 ('Movie1003', 'Movie0'): -0.5382970074196066,
 ('Movie0', 'Movie1004'): 0.2117003973573551,
 ('Movie1004', 'Movie0'): 0.2117003973573551,
 ('Movie0', 'Movie1005'): 0.01598278872731775,
 ('Movie1005', 'Movie0'): 0.01598278872731775,
 ('Movie0', 'Movie1006'): -0.35347260888802134,
 ('Movie1006', 'Movie0'): -0.35347260888802134,
 ('Movie0', 'Movie1007'): -0.5909613597773101,
 ('Movie1007', 'Movie0'

In [40]:
# Sort the similarity scores in desending order
sorted_similarity_scores = sorted(similarity_scores.items(), key=lambda item: item[1], reverse=True)
sorted_similarity_scores

[(('Movie1871', 'Movie1939'), 0.9719525414131447),
 (('Movie1939', 'Movie1871'), 0.9719525414131447),
 (('Movie1643', 'Movie1664'), 0.9481223556829746),
 (('Movie1664', 'Movie1643'), 0.9481223556829746),
 (('Movie1664', 'Movie1871'), 0.9456301211231214),
 (('Movie1871', 'Movie1664'), 0.9456301211231214),
 (('Movie1664', 'Movie1939'), 0.9421312876803896),
 (('Movie1939', 'Movie1664'), 0.9421312876803896),
 (('Movie1643', 'Movie1871'), 0.9325432665724349),
 (('Movie1871', 'Movie1643'), 0.9325432665724349),
 (('Movie1643', 'Movie1939'), 0.9227889031175537),
 (('Movie1939', 'Movie1643'), 0.9227889031175537),
 (('Movie251', 'Movie599'), 0.911811735037832),
 (('Movie599', 'Movie251'), 0.911811735037832),
 (('Movie363', 'Movie599'), 0.9104980952170446),
 (('Movie599', 'Movie363'), 0.9104980952170446),
 (('Movie251', 'Movie363'), 0.9092610017745625),
 (('Movie363', 'Movie251'), 0.9092610017745625),
 (('Movie1824', 'Movie1939'), 0.906519983310388),
 (('Movie1939', 'Movie1824'), 0.90651998331038

**Part 3**
---

$$
\text{$\hat{r}_u,_i$} = \mu_{i} + \frac{\sum_{j \in N_k(i)} sim(i, j) \left(r_{u, j} - \mu_{j} \right)}
{\sum_{j \in N_k(i)} |sim(i,j)|}
$$

where:  


- $\hat{r}_u,_i$: The predicted rating for the target user on item i.

- $\mu_{i}$: The average rating for the target item i computed from all users.

- $\mu_{j}$: The average rating for item j.

- $r_{u, j}$: The target user's rating for item j.

- $sim(i, j)$: The similarity between items i and j.

- $N_k(i)$: The set of the k most similar items to i that the target user has rated.

In [45]:
movie_means

{'Movie0': 4.256764342651847,
 'Movie1': 3.850799779371208,
 'Movie10': 3.851181102362205,
 'Movie100': 3.751237946312223,
 'Movie1000': 3.176598837209302,
 'Movie1001': 2.924611223799865,
 'Movie1002': 3.48471615720524,
 'Movie1003': 2.443353474320242,
 'Movie1004': 3.6027284001653577,
 'Movie1005': 3.4137995512341064,
 'Movie1006': 3.130359612724758,
 'Movie1007': 2.3751472320376914,
 'Movie1008': 2.124534161490683,
 'Movie1009': 3.745176174496644,
 'Movie101': 4.243207941483804,
 'Movie1010': 3.867021276595745,
 'Movie1011': 2.7632768361581923,
 'Movie1012': 3.600766283524904,
 'Movie1013': 3.849604966139955,
 'Movie1014': 2.345064724919094,
 'Movie1015': 4.026170105686965,
 'Movie1016': 3.593475073313783,
 'Movie1017': 4.116916709777548,
 'Movie1018': 3.08098223615465,
 'Movie1019': 3.681793625067531,
 'Movie102': 4.1580580397882825,
 'Movie1020': 3.5373427672955975,
 'Movie1021': 2.929807217004449,
 'Movie1022': 3.7121489621489623,
 'Movie1023': 3.8396039603960395,
 'Movie1024': 2

In [46]:
# k most similar users and to store predicted ratings for later
k = 25
predicted_ratings = {}

In [47]:
# Ranking all unrated movies for User25
for movie_i in target_user_unrated_movies:
    
    neighbors = []
    for movie_j in target_user_rated_movies:
        sim = similarity_scores.get((movie_i, movie_j)) or similarity_scores.get((movie_j, movie_i))
        if sim is not None:
            neighbors.append((movie_j, sim))

    neighbors = sorted(neighbors, key=lambda x: x[1], reverse=True)[:k]
    if not neighbors:
        continue

    mu_i = movie_means[movie_i]

    numerator = 0
    denominator = 0

    for movie_j, sim in neighbors:
        mu_j = movie_means[movie_j]
        r_uj = user_movie_to_rating[(target_user, movie_j)]
        numerator += sim * (r_uj - mu_j)
        denominator += abs(sim)

    predicted_rating = mu_i + (numerator / denominator if denominator != 0 else 0)
    predicted_ratings[movie_i] = predicted_rating

In [48]:
predicted_ratings

{'Movie1943': 3.482770468708223,
 'Movie1526': 3.263068036267937,
 'Movie1565': 3.3723997452198207,
 'Movie133': 3.8089319543946787,
 'Movie1280': 2.9800637710271976,
 'Movie1011': 3.1866573495020565,
 'Movie125': 3.102433047753518,
 'Movie1176': 3.338399684352318,
 'Movie1669': 3.474255879097386,
 'Movie1520': 3.0080933027256695,
 'Movie811': 3.5988252757180192,
 'Movie1767': 4.1666572759709375,
 'Movie1666': 3.5320280173163567,
 'Movie810': 3.1444670486635156,
 'Movie1341': 3.452615123897946,
 'Movie1836': 3.146188173525907,
 'Movie1103': 3.1741968251763515,
 'Movie1601': 3.904210873497568,
 'Movie1592': 3.141483132863888,
 'Movie319': 3.242775729223881,
 'Movie1906': 3.20749017580168,
 'Movie642': 4.140894797110174,
 'Movie1619': 2.705091879484206,
 'Movie390': 3.1780906030967655,
 'Movie1548': 3.016796677194717,
 'Movie1055': 3.596675532625941,
 'Movie1624': 3.764449936576715,
 'Movie494': 3.082574442254013,
 'Movie25': 3.6938234042157503,
 'Movie544': 3.8131284082698946,
 'Movie16

In [49]:
# Sort the predicted ratings in desending order for the top 5 movies
top5 = sorted(predicted_ratings.items(), key=lambda x: x[1], reverse=True)[:5]

In [50]:
print("Recommended movies for User25:-")
for movie, pred in top5:
    print(f'{movie} | predicted rating = {pred:.3f}')

Recommended movies for User25:-
Movie1350 | predicted rating = 4.436
Movie1352 | predicted rating = 4.290
Movie306 | predicted rating = 4.276
Movie263 | predicted rating = 4.230
Movie1494 | predicted rating = 4.190
